# Gold - ecommerce_itens_pedido

Notebook para criação de tabelas Gold em Delta Lake e réplica opcional para SQL Server/Azure, mantendo padrão de consumo analítico via Looker.

Este notebook assume que as tabelas Silver e `squad1.dq_monitoring_logs` já foram criadas em Delta.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SCHEMA = "squad1"
DQ_LOGS_TABLE = f"{SCHEMA}.dq_monitoring_logs"

# Ative como True somente se desejar replicar as Golds para SQL Server/Azure.
REPLICAR_SQLSERVER = False

# Caso use SQL Server, o notebook config precisa ter carregado:
# JDBC_HOSTNAME, JDBC_DATABASE, JDBC_USERNAME, JDBC_PASSWORD


In [0]:
def tabela_delta_existe(nome_tabela: str) -> bool:
    return spark.catalog.tableExists(nome_tabela)


def salvar_gold_delta(df, tabela_destino: str, chaves_merge=None):
    """
    Salva a Gold como tabela Delta gerenciada.
    Para Gold agregada, usamos overwrite para recalcular o snapshot analítico.
    Isso evita duplicidade mesmo com múltiplas execuções.
    """
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )
    print(f"Tabela Delta atualizada: {tabela_destino}")


def replicar_sqlserver(df, tabela_destino: str):
    """
    Replica para SQL Server usando overwrite, pois Gold agregada deve representar
    o estado analítico atual, não append incremental bruto.
    """
    if not REPLICAR_SQLSERVER:
        print(f"Réplica SQL Server desativada para {tabela_destino}")
        return

    (
        df.write
        .format("sqlserver")
        .mode("overwrite")
        .option("host", JDBC_HOSTNAME)
        .option("port", "1433")
        .option("database", JDBC_DATABASE)
        .option("user", JDBC_USERNAME)
        .option("password", JDBC_PASSWORD)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )
    print(f"Tabela replicada para SQL Server: {tabela_destino}")


def criar_gold_dq_resumo_por_regra(nome_tabela_silver: str):
    df_logs = spark.table(DQ_LOGS_TABLE)

    return (
        df_logs
        .filter(F.col("tabela") == nome_tabela_silver)
        .withColumn("data_referencia", F.to_date("timestamp_execucao"))
        .groupBy("data_referencia", "tabela", "regra", "severidade")
        .agg(
            F.sum("qtd_registros_falhos").alias("qtd_registros_falhos"),
            F.sum("qtd_registros_total").alias("qtd_registros_total")
        )
        .withColumn(
            "perc_falha",
            F.when(F.col("qtd_registros_total") > 0,
                   F.round((F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100, 2))
             .otherwise(F.lit(0.0))
        )
        .withColumn("gold_processed_at", F.current_timestamp())
    )


def criar_gold_dq_resumo_por_tabela(df_silver, nome_tabela_silver: str, colunas_falha: list):
    cond_falha = None
    for c in colunas_falha:
        expr = F.coalesce(F.col(c), F.lit(False)) == True
        cond_falha = expr if cond_falha is None else (cond_falha | expr)

    return (
        df_silver
        .withColumn("hora_referencia", F.date_trunc("hour", F.col("silver_processed_at")))
        .withColumn("linha_com_falha", cond_falha)
        .groupBy("hora_referencia")
        .agg(
            F.count("*").alias("qtd_registros_total"),
            F.sum(F.when(F.col("linha_com_falha"), 1).otherwise(0)).alias("qtd_registros_com_falha")
        )
        .withColumn("tabela", F.lit(nome_tabela_silver))
        .withColumn("qtd_registros_limpos", F.col("qtd_registros_total") - F.col("qtd_registros_com_falha"))
        .withColumn(
            "perc_registros_limpos",
            F.when(F.col("qtd_registros_total") > 0,
                   F.round((F.col("qtd_registros_limpos") / F.col("qtd_registros_total")) * 100, 2))
             .otherwise(F.lit(0.0))
        )
        .withColumn("gold_processed_at", F.current_timestamp())
        .select(
            "hora_referencia", "tabela", "qtd_registros_total",
            "qtd_registros_com_falha", "qtd_registros_limpos",
            "perc_registros_limpos", "gold_processed_at"
        )
    )


## Leitura das tabelas Silver e logs

In [0]:
SILVER_ITENS = "squad1.silver_ecommerce_itens_pedido"
SILVER_PEDIDOS = "squad1.silver_ecommerce_pedidos"
SILVER_PRODUTOS = "squad1.silver_ecommerce_produtos"  # opcional, caso exista

if not tabela_delta_existe(SILVER_ITENS):
    raise Exception(f"Tabela não encontrada: {SILVER_ITENS}")
if not tabela_delta_existe(DQ_LOGS_TABLE):
    raise Exception(f"Tabela não encontrada: {DQ_LOGS_TABLE}")

df_itens = spark.table(SILVER_ITENS)
df_pedidos = spark.table(SILVER_PEDIDOS) if tabela_delta_existe(SILVER_PEDIDOS) else None

display(df_itens.limit(5))

## Gold DQ - resumo por regra e por tabela

In [0]:
colunas_falha_itens = [
    "r1_id_item_pedido_falhou",
    "r2_id_pedido_fk_falhou",
    "r3_sku_fk_falhou",
    "r4_quantidade_falhou",
    "r5_preco_unitario_falhou",
    "r6_desconto_maior_preco_falhou",
    "r7_total_pedido_divergente_falhou",
    "r8_desconto_negativo_falhou",
    "r9_percentual_desconto_falhou",
    "r10_pedido_sem_item_falhou"
]

# Mantém somente colunas que existem no DataFrame, para evitar erro se o nome no notebook Silver variar.
colunas_falha_itens = [c for c in colunas_falha_itens if c in df_itens.columns]

if len(colunas_falha_itens) == 0:
    raise Exception("Nenhuma coluna de falha encontrada na Silver de itens_pedido.")

df_gold_dq_regra_itens = criar_gold_dq_resumo_por_regra("silver_ecommerce_itens_pedido")
df_gold_dq_tabela_itens = criar_gold_dq_resumo_por_tabela(df_itens, "silver_ecommerce_itens_pedido", colunas_falha_itens)

salvar_gold_delta(df_gold_dq_regra_itens, "squad1.gold_itens_pedido_dq_resumo_por_regra")
salvar_gold_delta(df_gold_dq_tabela_itens, "squad1.gold_itens_pedido_dq_resumo_por_tabela")

replicar_sqlserver(df_gold_dq_regra_itens, "squad1.gold_itens_pedido_dq_resumo_por_regra")
replicar_sqlserver(df_gold_dq_tabela_itens, "squad1.gold_itens_pedido_dq_resumo_por_tabela")

## KPIs específicos de itens de pedido

In [0]:
df_itens_kpi_base = (
    df_itens
    .withColumn("receita_bruta_item", F.col("preco_unitario") * F.col("quantidade"))
    .withColumn("desconto_total_item", F.coalesce(F.col("desconto_aplicado"), F.lit(0.0)) * F.col("quantidade"))
    .withColumn("receita_liquida_item", (F.col("preco_unitario") - F.coalesce(F.col("desconto_aplicado"), F.lit(0.0))) * F.col("quantidade"))
)

if "silver_processed_at" in df_itens_kpi_base.columns:
    df_itens_kpi_base = df_itens_kpi_base.withColumn("data_referencia", F.to_date("silver_processed_at"))
elif "bronze_ingested_at" in df_itens_kpi_base.columns:
    df_itens_kpi_base = df_itens_kpi_base.withColumn("data_referencia", F.to_date("bronze_ingested_at"))
else:
    df_itens_kpi_base = df_itens_kpi_base.withColumn("data_referencia", F.current_date())

df_gold_itens_kpis = (
    df_itens_kpi_base
    .groupBy("data_referencia")
    .agg(
        F.count("*").alias("qtd_itens"),
        F.countDistinct("id_pedido").alias("qtd_pedidos"),
        F.countDistinct("sku").alias("qtd_skus"),
        F.sum("quantidade").alias("qtd_unidades_vendidas"),
        F.round(F.sum("receita_bruta_item"), 2).alias("receita_bruta"),
        F.round(F.sum("desconto_total_item"), 2).alias("desconto_total"),
        F.round(F.sum("receita_liquida_item"), 2).alias("receita_liquida"),
        F.round(F.avg("preco_unitario"), 2).alias("preco_unitario_medio"),
        F.round(F.avg("desconto_aplicado"), 2).alias("desconto_medio")
    )
    .withColumn("ticket_medio_item", F.round(F.col("receita_liquida") / F.col("qtd_itens"), 2))
    .withColumn("perc_desconto_sobre_receita", F.round((F.col("desconto_total") / F.col("receita_bruta")) * 100, 2))
    .withColumn("gold_processed_at", F.current_timestamp())
)

salvar_gold_delta(df_gold_itens_kpis, "squad1.gold_itens_pedido_kpis")
replicar_sqlserver(df_gold_itens_kpis, "squad1.gold_itens_pedido_kpis")

display(df_gold_itens_kpis)

##  Rankings para Looker

In [0]:
df_gold_itens_top_skus = (
    df_itens_kpi_base
    .groupBy("sku")
    .agg(
        F.sum("quantidade").alias("qtd_unidades_vendidas"),
        F.countDistinct("id_pedido").alias("qtd_pedidos"),
        F.round(F.sum("receita_liquida_item"), 2).alias("receita_liquida"),
        F.round(F.sum("desconto_total_item"), 2).alias("desconto_total")
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("receita_liquida"))
)

salvar_gold_delta(df_gold_itens_top_skus, "squad1.gold_itens_pedido_top_skus")
replicar_sqlserver(df_gold_itens_top_skus, "squad1.gold_itens_pedido_top_skus")

display(df_gold_itens_top_skus.limit(20))

In [0]:
tabelas_gold_criadas = ['squad1.gold_itens_pedido_dq_resumo_por_regra', 'squad1.gold_itens_pedido_dq_resumo_por_tabela', 'squad1.gold_itens_pedido_kpis', 'squad1.gold_itens_pedido_top_skus']

In [0]:
# Validação final das tabelas criadas
for tabela in tabelas_gold_criadas:
    print(f"\n{tabela}")
    spark.sql(f"DESCRIBE DETAIL {tabela}").select("format", "numFiles", "sizeInBytes", "location").show(truncate=False)
    print(f"Registros: {spark.table(tabela).count()}")
    display(spark.table(tabela).limit(10))
